# Khám phá và Làm sạch Dữ liệu - BAF Base Dataset

Thống kê tổng quan dữ liệu thô, xử lý giá trị sentinel -1 đại diện cho dữ liệu thiếu, tối ưu kiểu dữ liệu và xuất sang định dạng Parquet.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import duckdb

sys.path.append("..")
from src.data.load_data import load_raw_dataset
from src.features.clean_and_engineer import clean_and_transform_pipeline


### 1. Thống kê nhanh dữ liệu thô qua DuckDB

In [ ]:
conn = duckdb.connect()
conn.execute("""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(fraud_bool) AS fraud_count,
        ROUND(AVG(fraud_bool) * 100, 3) AS fraud_rate_pct,
        COUNT(DISTINCT month) AS num_months
    FROM "../data/raw/Base.csv"
""").df()

### 2. Kiểm tra tỷ lệ dữ liệu thiếu mã hóa bằng sentinel -1
Tập dữ liệu BAF sử dụng -1 để thể hiện các trường không có dữ liệu ghi nhận.

In [ ]:
conn.execute("""
    SELECT 
        COUNT(*) AS total,
        ROUND(COUNT(CASE WHEN prev_address_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS prev_address_missing_pct,
        ROUND(COUNT(CASE WHEN intended_balcon_amount < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS intended_balcon_missing_pct,
        ROUND(COUNT(CASE WHEN bank_months_count = -1 THEN 1 END) * 100.0 / COUNT(*), 2) AS bank_months_missing_pct,
        ROUND(COUNT(CASE WHEN session_length_in_minutes < 0 THEN 1 END) * 100.0 / COUNT(*), 2) AS session_length_missing_pct
    FROM "../data/raw/Base.csv"
""").df()

### 3. Thực thi pipeline làm sạch và lưu sang Parquet

In [ ]:
raw_df = load_raw_dataset("Base.csv")
clean_df = clean_and_transform_pipeline(raw_df)

out_path = Path("../data/processed/clean_base.parquet")
clean_df.to_parquet(out_path, index=False, engine="pyarrow")
print(f"Đã lưu tệp kết quả tại {out_path} ({out_path.stat().st_size / (1024**2):.1f} MB)")